# Open-Set Reliability Improvements: External Evaluation

This notebook strengthens the thesis without training another model. It uses the saved
ResNet-50 detector to perform four linked analyses:

1. a larger balanced evaluation on MS COCOAI/Defactify;
2. paired stratified bootstrap confidence intervals before and after temperature scaling;
3. class-conditional error attribution separating MS COCO false positives from
   generator-specific false negatives; and
4. an illustrative cybersecurity allow/review/escalate policy simulation.

**Open-set scope.** The semantic labels remain `real` and `AI-generated`. The source
space is open because the external generators were absent from model development.
This is an external open-generator reliability stress test, not an unknown-class
recognition system.

**No training occurs in this notebook.** The external labels are used only for evaluation.
The saved validation temperature is transferred unchanged; it is not refitted here.

Recommended first run: Colab GPU with `MAX_PER_SOURCE = 1000`. After that run succeeds,
set `MAX_PER_SOURCE = None` for the full external test split if time and storage permit.


In [ ]:
%pip -q install "datasets>=2.19,<4" pyarrow


In [ ]:
import io
import json
import math
import os
import platform
import shutil
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sklearn
import tensorflow as tf
from datasets import Image as HFImage
from datasets import load_dataset
from IPython.display import display
from PIL import Image
from scipy.special import expit, logit, softmax
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    log_loss,
    roc_auc_score,
)
from tqdm.auto import tqdm

# -----------------------------
# Edit only this configuration
# -----------------------------
USE_GOOGLE_DRIVE = True
DATASET_ID = "Rajarshi-Roy-research/Defactify_Image_Dataset"
SPLIT_TO_USE = "test"

# Recommended first run: 1000. Use None only after the first run succeeds.
MAX_PER_SOURCE = 1000
RANDOM_SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32
CHECKPOINT_EVERY_BATCHES = 25

# Final validation-derived temperature from phase 3. Do not fit on external labels.
TEMPERATURE = 1.2408854103475755
HIGH_CONF_THRESHOLD = 0.90
N_BOOTSTRAP = 1000

# Illustrative policy thresholds, not production-optimised thresholds.
POLICY_ALLOW_MAX = 0.20
POLICY_ESCALATE_MIN = 0.80
DOWNLOAD_ZIP_AT_END = True

SOURCE_MAP = {
    0: "Real MS COCO",
    1: "Stable Diffusion 2.1",
    2: "Stable Diffusion XL",
    3: "Stable Diffusion 3",
    4: "DALL-E 3",
    5: "Midjourney v6",
}

if USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/OpenSet_Thesis_Improvement")
else:
    BASE_DIR = Path("/content/OpenSet_Thesis_Improvement")

MODEL_DIR = BASE_DIR / "model"
MODEL_PATH = MODEL_DIR / "resnet50_finetuned.keras"
subset_tag = "full" if MAX_PER_SOURCE is None else f"max{MAX_PER_SOURCE}_per_source"
RUN_DIR = BASE_DIR / "results" / f"{SPLIT_TO_USE}_{subset_tag}_seed{RANDOM_SEED}"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)

config = {
    "dataset_id": DATASET_ID,
    "split": SPLIT_TO_USE,
    "max_per_source": MAX_PER_SOURCE,
    "random_seed": RANDOM_SEED,
    "image_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "temperature": TEMPERATURE,
    "high_confidence_threshold": HIGH_CONF_THRESHOLD,
    "n_bootstrap": N_BOOTSTRAP,
    "policy_allow_max": POLICY_ALLOW_MAX,
    "policy_escalate_min": POLICY_ESCALATE_MIN,
}
(RUN_DIR / "run_config.json").write_text(json.dumps(config, indent=2))

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))
print("Persistent base directory:", BASE_DIR)
print("Run directory:", RUN_DIR)


## 1. Locate the saved model

On the first run, this cell asks for `resnet50_finetuned.keras` and copies it to the
persistent Drive folder. Later runs reuse the Drive copy.


In [ ]:
if not MODEL_PATH.exists():
    from google.colab import files

    print("Upload resnet50_finetuned.keras")
    uploaded = files.upload()
    keras_files = [Path(name) for name in uploaded if name.lower().endswith(".keras")]
    if len(keras_files) != 1:
        raise RuntimeError("Upload exactly one .keras model file.")
    shutil.copy2(keras_files[0], MODEL_PATH)

print("Using model:", MODEL_PATH)
model = tf.keras.models.load_model(str(MODEL_PATH), compile=False)
print("Model input shape:", model.input_shape)
print("Model output shape:", model.output_shape)


## 2. Load the recent external dataset and create a reproducible sample

Sampling is stratified by source. The selected row identifiers are stored in Drive so
the same sample is reused after a restart.


In [ ]:
def find_column(columns, candidates, required=True):
    lookup = {name.lower(): name for name in columns}
    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]
    if required:
        raise KeyError(f"Could not find any of {candidates} in {columns}")
    return None


print(f"Loading {DATASET_ID}, split={SPLIT_TO_USE!r}")
ds = load_dataset(DATASET_ID, split=SPLIT_TO_USE, cache_dir="/content/hf_cache")

IMAGE_COL = find_column(ds.column_names, ["Image", "image"])
CAPTION_COL = find_column(
    ds.column_names, ["Caption", "caption", "prompt", "text"], required=False
)
LABEL_COL = find_column(ds.column_names, ["Label_A", "label_a", "label", "binary_label"])
SOURCE_COL = find_column(ds.column_names, ["Label_B", "label_b", "source", "source_label"])
ds = ds.cast_column(IMAGE_COL, HFImage(decode=False))

labels = np.asarray(ds[LABEL_COL], dtype=int)
sources = np.asarray(ds[SOURCE_COL], dtype=int)

source_counts = (
    pd.DataFrame({"source_id": sources, "label_binary": labels})
    .groupby(["source_id", "label_binary"])
    .size()
    .rename("available_rows")
    .reset_index()
)
source_counts["source_name"] = source_counts["source_id"].map(SOURCE_MAP)
display(source_counts)

# Dataset integrity: source 0 is real and every other source is AI-generated.
assert set(np.unique(labels[sources == 0])) == {0}
assert set(np.unique(labels[sources != 0])) == {1}

manifest_path = RUN_DIR / "selected_manifest.csv"
if manifest_path.exists():
    manifest = pd.read_csv(manifest_path)
    selected_indices = manifest["original_index"].astype(int).tolist()
    print("Reusing saved manifest:", manifest_path)
else:
    rng = np.random.default_rng(RANDOM_SEED)
    selected_indices = []
    for source_id in sorted(np.unique(sources)):
        idx = np.flatnonzero(sources == source_id)
        rng.shuffle(idx)
        if MAX_PER_SOURCE is not None:
            idx = idx[: min(MAX_PER_SOURCE, len(idx))]
        selected_indices.extend(int(i) for i in idx)
    rng.shuffle(selected_indices)
    manifest = pd.DataFrame(
        {
            "original_index": selected_indices,
            "label_binary": labels[selected_indices],
            "source_id": sources[selected_indices],
        }
    )
    manifest["source_name"] = manifest["source_id"].map(SOURCE_MAP)
    manifest.to_csv(manifest_path, index=False)
    print("Saved manifest:", manifest_path)

print("Selected rows:", len(selected_indices))
display(manifest.groupby(["source_id", "source_name", "label_binary"]).size().reset_index(name="n"))


## 3. Run resumable inference

Predictions are checkpointed to Drive every 25 batches. Re-running this cell skips rows
already present in the checkpoint or final prediction file.


In [ ]:
def image_to_pil(image_item):
    if isinstance(image_item, Image.Image):
        return image_item
    if isinstance(image_item, dict):
        if image_item.get("bytes") is not None:
            return Image.open(io.BytesIO(image_item["bytes"]))
        if image_item.get("path"):
            return Image.open(image_item["path"])
    if isinstance(image_item, (str, os.PathLike)):
        return Image.open(image_item)
    raise TypeError(f"Unsupported image type: {type(image_item)}")


def preprocess_batch(pil_images):
    images = []
    for image in pil_images:
        resized = image.convert("RGB").resize((IMG_SIZE, IMG_SIZE))
        images.append(np.asarray(resized, dtype=np.float32))
    batch = np.stack(images, axis=0)
    return tf.keras.applications.resnet50.preprocess_input(batch)


def normalize_model_output(raw_output):
    raw = np.asarray(raw_output)
    if raw.ndim == 2 and raw.shape[1] == 2:
        row_sums = raw.sum(axis=1)
        looks_like_probability = (
            np.all(raw >= 0)
            and np.all(raw <= 1)
            and np.allclose(row_sums, 1.0, atol=1e-3)
        )
        if looks_like_probability:
            probability = np.clip(raw[:, 1], 1e-6, 1 - 1e-6)
            return probability, logit(probability), "2-class probabilities"
        probability = softmax(raw, axis=1)[:, 1]
        return probability, raw[:, 1] - raw[:, 0], "2-class logits"

    raw = raw.reshape(-1)
    looks_like_probability = np.all(raw >= 0) and np.all(raw <= 1)
    if looks_like_probability:
        probability = np.clip(raw, 1e-6, 1 - 1e-6)
        return probability, logit(probability), "1-class probability"
    probability = expit(raw.astype(float))
    return probability, raw.astype(float), "1-class logit"


checkpoint_path = RUN_DIR / "predictions_checkpoint.csv"
predictions_path = RUN_DIR / "external_predictions.csv"

if predictions_path.exists():
    existing = pd.read_csv(predictions_path)
    print("Loaded final prediction file:", predictions_path)
elif checkpoint_path.exists():
    existing = pd.read_csv(checkpoint_path)
    print("Resuming checkpoint:", checkpoint_path)
else:
    existing = pd.DataFrame()

completed = set(existing.get("original_index", pd.Series(dtype=int)).astype(int).tolist())
remaining = [idx for idx in selected_indices if idx not in completed]
print(f"Completed: {len(completed)} | Remaining: {len(remaining)}")

frames = [existing] if not existing.empty else []
pending_frames = []
output_kinds = set(existing.get("model_output_kind", pd.Series(dtype=str)).dropna().tolist())

for batch_number, start in enumerate(
    tqdm(range(0, len(remaining), BATCH_SIZE), desc="External inference"), start=1
):
    batch_indices = remaining[start : start + BATCH_SIZE]
    rows = [ds[i] for i in batch_indices]
    batch_images = [image_to_pil(row[IMAGE_COL]) for row in rows]
    batch_x = preprocess_batch(batch_images)
    raw_output = model(batch_x, training=False).numpy()
    prob_ai, binary_logit, output_kind = normalize_model_output(raw_output)
    output_kinds.add(output_kind)

    records = []
    for offset, (original_idx, row) in enumerate(zip(batch_indices, rows)):
        source_id = int(row[SOURCE_COL])
        label_binary = int(row[LABEL_COL])
        records.append(
            {
                "dataset_id": DATASET_ID,
                "split": SPLIT_TO_USE,
                "original_index": int(original_idx),
                "label_binary": label_binary,
                "source_id": source_id,
                "source_name": SOURCE_MAP.get(source_id, f"source_{source_id}"),
                "prob_ai": float(prob_ai[offset]),
                "binary_logit": float(binary_logit[offset]),
                "model_output_kind": output_kind,
            }
        )
    pending_frames.append(pd.DataFrame(records))

    if batch_number % CHECKPOINT_EVERY_BATCHES == 0:
        frames.extend(pending_frames)
        pending_frames = []
        checkpoint = (
            pd.concat(frames, ignore_index=True)
            .drop_duplicates("original_index", keep="last")
            .sort_values("original_index")
        )
        checkpoint.to_csv(checkpoint_path, index=False)

frames.extend(pending_frames)
predictions = (
    pd.concat(frames, ignore_index=True)
    .drop_duplicates("original_index", keep="last")
    .sort_values("original_index")
    .reset_index(drop=True)
)
expected = set(int(i) for i in selected_indices)
observed = set(predictions["original_index"].astype(int))
if observed != expected:
    raise RuntimeError(
        f"Prediction rows do not match manifest: expected {len(expected)}, found {len(observed)}"
    )

predictions["prob_ai"] = predictions["prob_ai"].clip(1e-6, 1 - 1e-6)
predictions["prob_ai_temp"] = expit(
    predictions["binary_logit"].astype(float) / float(TEMPERATURE)
)
predictions["pred_label"] = (predictions["prob_ai"] >= 0.5).astype(int)
predictions["pred_label_temp"] = (predictions["prob_ai_temp"] >= 0.5).astype(int)
predictions["confidence"] = np.maximum(predictions["prob_ai"], 1 - predictions["prob_ai"])
predictions["confidence_temp"] = np.maximum(
    predictions["prob_ai_temp"], 1 - predictions["prob_ai_temp"]
)
predictions["correct"] = predictions["pred_label"] == predictions["label_binary"]
predictions["correct_temp"] = predictions["pred_label_temp"] == predictions["label_binary"]
assert np.array_equal(predictions["pred_label"], predictions["pred_label_temp"])

predictions.to_csv(predictions_path, index=False)
if checkpoint_path.exists():
    checkpoint_path.unlink()
print("Model output kind(s):", sorted(output_kinds))
print("Saved predictions:", predictions_path)
display(predictions.head())


## 4. Harmonised metrics

This notebook uses top-label ECE with 15 bins for both raw and scaled probabilities.
Absolute ECE values should be compared only within this external protocol.


In [ ]:
def top_label_ece(y_true, prob_ai, n_bins=15):
    y_true = np.asarray(y_true, dtype=int)
    prob_ai = np.asarray(prob_ai, dtype=float)
    pred = (prob_ai >= 0.5).astype(int)
    confidence = np.maximum(prob_ai, 1 - prob_ai)
    correct = (pred == y_true).astype(float)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for index in range(n_bins):
        if index == 0:
            mask = (confidence >= edges[index]) & (confidence <= edges[index + 1])
        else:
            mask = (confidence > edges[index]) & (confidence <= edges[index + 1])
        if np.any(mask):
            ece += mask.mean() * abs(correct[mask].mean() - confidence[mask].mean())
    return float(ece)


def compute_metrics(frame, prob_col):
    y_true = frame["label_binary"].astype(int).to_numpy()
    probability = np.clip(frame[prob_col].astype(float).to_numpy(), 1e-6, 1 - 1e-6)
    pred = (probability >= 0.5).astype(int)
    confidence = np.maximum(probability, 1 - probability)
    wrong = pred != y_true
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    high = confidence >= HIGH_CONF_THRESHOLD
    return {
        "n": int(len(frame)),
        "accuracy": float(accuracy_score(y_true, pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, pred)),
        "auroc": float(roc_auc_score(y_true, probability)),
        "log_loss": float(log_loss(y_true, probability, labels=[0, 1])),
        "brier": float(brier_score_loss(y_true, probability)),
        "top_label_ece_15": top_label_ece(y_true, probability, n_bins=15),
        "avg_conf_wrong": float(confidence[wrong].mean()) if np.any(wrong) else 0.0,
        "high_confidence_errors": int(np.sum(wrong & high)),
        "hce_rate_all": float(np.mean(wrong & high)),
        "hce_rate_high": float(np.sum(wrong & high) / np.sum(high)) if np.any(high) else 0.0,
        "specificity_real": float(tn / (tn + fp)),
        "ai_recall": float(tp / (tp + fn)),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


overall_rows = []
for state, prob_col in [("Raw", "prob_ai"), ("Temperature-scaled", "prob_ai_temp")]:
    row = compute_metrics(predictions, prob_col)
    row["state"] = state
    row["probability_column"] = prob_col
    overall_rows.append(row)
overall_metrics = pd.DataFrame(overall_rows)
overall_metrics.to_csv(RUN_DIR / "overall_metrics.csv", index=False)
display(overall_metrics)


## 5. Paired stratified bootstrap confidence intervals

Rows are resampled within each source, preserving the source composition. The same
bootstrap sample is used for raw and scaled probabilities, making the difference paired.


In [ ]:
BOOTSTRAP_METRICS = [
    "accuracy",
    "balanced_accuracy",
    "auroc",
    "log_loss",
    "brier",
    "top_label_ece_15",
    "avg_conf_wrong",
    "hce_rate_all",
    "hce_rate_high",
    "specificity_real",
    "ai_recall",
]

source_positions = {
    int(source_id): np.flatnonzero(predictions["source_id"].to_numpy() == source_id)
    for source_id in sorted(predictions["source_id"].unique())
}
rng = np.random.default_rng(RANDOM_SEED + 1000)
raw_bootstrap = []
scaled_bootstrap = []

for replicate in tqdm(range(N_BOOTSTRAP), desc="Paired stratified bootstrap"):
    sampled_positions = np.concatenate(
        [rng.choice(pos, size=len(pos), replace=True) for pos in source_positions.values()]
    )
    sample = predictions.iloc[sampled_positions]
    raw_bootstrap.append(compute_metrics(sample, "prob_ai"))
    scaled_bootstrap.append(compute_metrics(sample, "prob_ai_temp"))

raw_bootstrap = pd.DataFrame(raw_bootstrap)
scaled_bootstrap = pd.DataFrame(scaled_bootstrap)
raw_bootstrap.to_csv(RUN_DIR / "bootstrap_raw_replicates.csv", index=False)
scaled_bootstrap.to_csv(RUN_DIR / "bootstrap_scaled_replicates.csv", index=False)

ci_rows = []
delta_rows = []
point_raw = compute_metrics(predictions, "prob_ai")
point_scaled = compute_metrics(predictions, "prob_ai_temp")
for metric in BOOTSTRAP_METRICS:
    for state, point, values in [
        ("Raw", point_raw[metric], raw_bootstrap[metric]),
        ("Temperature-scaled", point_scaled[metric], scaled_bootstrap[metric]),
    ]:
        ci_rows.append(
            {
                "metric": metric,
                "state": state,
                "estimate": point,
                "ci_2_5": float(values.quantile(0.025)),
                "ci_97_5": float(values.quantile(0.975)),
            }
        )
    delta = scaled_bootstrap[metric] - raw_bootstrap[metric]
    delta_rows.append(
        {
            "metric": metric,
            "raw_estimate": point_raw[metric],
            "scaled_estimate": point_scaled[metric],
            "paired_delta_scaled_minus_raw": point_scaled[metric] - point_raw[metric],
            "delta_ci_2_5": float(delta.quantile(0.025)),
            "delta_ci_97_5": float(delta.quantile(0.975)),
        }
    )

bootstrap_ci = pd.DataFrame(ci_rows)
bootstrap_delta = pd.DataFrame(delta_rows)
bootstrap_ci.to_csv(RUN_DIR / "bootstrap_95ci.csv", index=False)
bootstrap_delta.to_csv(RUN_DIR / "bootstrap_paired_deltas.csv", index=False)
display(bootstrap_ci)
display(bootstrap_delta)


## 6. Class-conditional external error attribution

This does not claim to causally isolate every source of dataset shift. It answers the
safer question: how much of the observed external failure appears as false positives on
changed real-image provenance, and how much appears as misses for each newer generator?


In [ ]:
def wilson_interval(successes, total, z=1.959963984540054):
    if total == 0:
        return np.nan, np.nan
    p = successes / total
    denominator = 1 + z**2 / total
    centre = (p + z**2 / (2 * total)) / denominator
    margin = z * math.sqrt((p * (1 - p) + z**2 / (4 * total)) / total) / denominator
    return max(0.0, centre - margin), min(1.0, centre + margin)


error_rows = []
for source_id, source_frame in predictions.groupby("source_id", sort=True):
    source_id = int(source_id)
    y_true = source_frame["label_binary"].astype(int).to_numpy()
    pred = source_frame["pred_label"].astype(int).to_numpy()
    if source_id == 0:
        errors = int(np.sum(pred == 1))
        error_name = "False-positive rate on real images"
    else:
        errors = int(np.sum(pred == 0))
        error_name = "False-negative rate on generated images"
    n = len(source_frame)
    lower, upper = wilson_interval(errors, n)
    wrong = pred != y_true
    error_rows.append(
        {
            "source_id": source_id,
            "source_name": SOURCE_MAP[source_id],
            "error_type": error_name,
            "n": n,
            "errors": errors,
            "error_rate": errors / n,
            "error_rate_ci_2_5": lower,
            "error_rate_ci_97_5": upper,
            "mean_raw_confidence_on_errors": float(
                source_frame.loc[wrong, "confidence"].mean()
            ) if np.any(wrong) else 0.0,
            "mean_scaled_confidence_on_errors": float(
                source_frame.loc[wrong, "confidence_temp"].mean()
            ) if np.any(wrong) else 0.0,
        }
    )

class_conditional = pd.DataFrame(error_rows)
class_conditional.to_csv(RUN_DIR / "class_conditional_error_attribution.csv", index=False)
display(class_conditional)

# Pair each AI generator with the same MS COCO real subset.
real_rows = predictions[predictions["source_id"] == 0]
generator_pair_rows = []
for source_id in sorted(i for i in predictions["source_id"].unique() if int(i) != 0):
    paired = pd.concat(
        [real_rows, predictions[predictions["source_id"] == int(source_id)]],
        ignore_index=True,
    )
    for state, prob_col in [("Raw", "prob_ai"), ("Temperature-scaled", "prob_ai_temp")]:
        metrics = compute_metrics(paired, prob_col)
        metrics.update(
            {
                "source_id": int(source_id),
                "source_name": SOURCE_MAP[int(source_id)],
                "state": state,
            }
        )
        generator_pair_rows.append(metrics)
generator_pairs = pd.DataFrame(generator_pair_rows)
generator_pairs.to_csv(RUN_DIR / "generator_pair_metrics.csv", index=False)

fig, ax = plt.subplots(figsize=(8.5, 4.8))
plot_frame = class_conditional.sort_values("source_id")
y = np.arange(len(plot_frame))
x = plot_frame["error_rate"].to_numpy()
lower = x - plot_frame["error_rate_ci_2_5"].to_numpy()
upper = plot_frame["error_rate_ci_97_5"].to_numpy() - x
colours = ["#C44E52" if sid == 0 else "#3B6EA8" for sid in plot_frame["source_id"]]
ax.errorbar(x, y, xerr=np.vstack([lower, upper]), fmt="none", ecolor="#4A4A4A", capsize=4)
ax.scatter(x, y, s=70, c=colours, zorder=3)
ax.set_yticks(y, plot_frame["source_name"])
ax.set_xlabel("Class-conditional error rate with Wilson 95% interval")
ax.set_title("External failure attribution by real source and generator")
ax.set_xlim(left=0)
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
fig.savefig(RUN_DIR / "class_conditional_error_forest.png", dpi=220)
plt.show()


## 7. Cybersecurity allow/review/escalate simulation

This is a descriptive policy simulation, not a recommendation to block users. `Escalate`
means route to corroboration or human review. The thresholds are fixed in the configuration
and are not optimised on external test labels.


In [ ]:
def policy_metrics(frame, prob_col, allow_max, escalate_min):
    probability = frame[prob_col].astype(float).to_numpy()
    y_true = frame["label_binary"].astype(int).to_numpy()
    allow = probability < allow_max
    escalate = probability > escalate_min
    review = ~(allow | escalate)
    automatic = allow | escalate
    correct_automatic = (allow & (y_true == 0)) | (escalate & (y_true == 1))
    false_allow = allow & (y_true == 1)
    false_escalate = escalate & (y_true == 0)
    real = y_true == 0
    ai = y_true == 1
    return {
        "n": len(frame),
        "allow_rate": float(allow.mean()),
        "review_rate": float(review.mean()),
        "escalate_rate": float(escalate.mean()),
        "automatic_coverage": float(automatic.mean()),
        "automatic_accuracy": float(correct_automatic.sum() / automatic.sum())
        if automatic.any()
        else np.nan,
        "automatic_error_rate": float(1 - correct_automatic.sum() / automatic.sum())
        if automatic.any()
        else np.nan,
        "wrong_automatic_decisions": int((automatic & ~correct_automatic).sum()),
        "wrong_automatic_per_1000": float(1000 * (automatic & ~correct_automatic).mean()),
        "false_allow_ai_count": int(false_allow.sum()),
        "false_allow_rate_among_ai": float(false_allow.sum() / ai.sum()),
        "false_escalate_real_count": int(false_escalate.sum()),
        "false_escalate_rate_among_real": float(false_escalate.sum() / real.sum()),
    }


policy_rows = []
for state, prob_col in [("Raw", "prob_ai"), ("Temperature-scaled", "prob_ai_temp")]:
    row = policy_metrics(
        predictions, prob_col, POLICY_ALLOW_MAX, POLICY_ESCALATE_MIN
    )
    row.update(
        {
            "state": state,
            "allow_max": POLICY_ALLOW_MAX,
            "escalate_min": POLICY_ESCALATE_MIN,
        }
    )
    policy_rows.append(row)
policy_summary = pd.DataFrame(policy_rows)
policy_summary.to_csv(RUN_DIR / "cybersecurity_policy_summary.csv", index=False)
display(policy_summary)

# Bootstrap the fixed policy using the same source-stratified design.
policy_bootstrap_rows = []
policy_rng = np.random.default_rng(RANDOM_SEED + 2000)
for replicate in tqdm(range(N_BOOTSTRAP), desc="Policy bootstrap"):
    sampled_positions = np.concatenate(
        [policy_rng.choice(pos, size=len(pos), replace=True) for pos in source_positions.values()]
    )
    sample = predictions.iloc[sampled_positions]
    for state, prob_col in [("Raw", "prob_ai"), ("Temperature-scaled", "prob_ai_temp")]:
        row = policy_metrics(sample, prob_col, POLICY_ALLOW_MAX, POLICY_ESCALATE_MIN)
        row.update({"replicate": replicate, "state": state})
        policy_bootstrap_rows.append(row)
policy_bootstrap = pd.DataFrame(policy_bootstrap_rows)
policy_bootstrap.to_csv(RUN_DIR / "cybersecurity_policy_bootstrap_replicates.csv", index=False)

policy_ci_rows = []
policy_metric_names = [
    "review_rate",
    "automatic_coverage",
    "automatic_error_rate",
    "wrong_automatic_per_1000",
    "false_allow_rate_among_ai",
    "false_escalate_rate_among_real",
]
for state in ["Raw", "Temperature-scaled"]:
    state_boot = policy_bootstrap[policy_bootstrap["state"] == state]
    point = policy_summary[policy_summary["state"] == state].iloc[0]
    for metric in policy_metric_names:
        policy_ci_rows.append(
            {
                "state": state,
                "metric": metric,
                "estimate": point[metric],
                "ci_2_5": float(state_boot[metric].quantile(0.025)),
                "ci_97_5": float(state_boot[metric].quantile(0.975)),
            }
        )
policy_ci = pd.DataFrame(policy_ci_rows)
policy_ci.to_csv(RUN_DIR / "cybersecurity_policy_95ci.csv", index=False)
display(policy_ci)

sweep_rows = []
for threshold in np.linspace(0.50, 0.99, 50):
    for state, prob_col in [("Raw", "prob_ai"), ("Temperature-scaled", "prob_ai_temp")]:
        row = policy_metrics(predictions, prob_col, 1 - threshold, threshold)
        row.update({"state": state, "symmetric_confidence_threshold": threshold})
        sweep_rows.append(row)
policy_sweep = pd.DataFrame(sweep_rows)
policy_sweep.to_csv(RUN_DIR / "cybersecurity_policy_threshold_sweep.csv", index=False)

fig, ax = plt.subplots(figsize=(7.5, 5.2))
for state, colour in [("Raw", "#C44E52"), ("Temperature-scaled", "#2A7F62")]:
    part = policy_sweep[policy_sweep["state"] == state]
    ax.plot(
        part["automatic_coverage"],
        part["automatic_error_rate"],
        marker="o",
        markersize=3,
        linewidth=1.8,
        label=state,
        color=colour,
    )
ax.set_xlabel("Automatic decision coverage")
ax.set_ylabel("Error rate among automatic decisions")
ax.set_title("Cybersecurity review trade-off under confidence thresholds")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(RUN_DIR / "cybersecurity_risk_coverage_curve.png", dpi=220)
plt.show()


## 8. Calibration figures and paired-effect view


In [ ]:
def reliability_points(frame, prob_col, n_bins=15):
    y_true = frame["label_binary"].astype(int).to_numpy()
    probability = frame[prob_col].astype(float).to_numpy()
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    x, y, n = [], [], []
    for index in range(n_bins):
        if index == 0:
            mask = (probability >= edges[index]) & (probability <= edges[index + 1])
        else:
            mask = (probability > edges[index]) & (probability <= edges[index + 1])
        if np.any(mask):
            x.append(float(probability[mask].mean()))
            y.append(float(y_true[mask].mean()))
            n.append(int(mask.sum()))
    return np.asarray(x), np.asarray(y), np.asarray(n)


fig, ax = plt.subplots(figsize=(6.2, 5.5))
ax.plot([0, 1], [0, 1], linestyle="--", color="#6B6B6B", linewidth=1.2, label="Ideal")
for state, prob_col, colour in [
    ("Raw", "prob_ai", "#C44E52"),
    ("Temperature-scaled", "prob_ai_temp", "#2A7F62"),
]:
    x, y, n = reliability_points(predictions, prob_col)
    ax.plot(x, y, marker="o", linewidth=1.8, label=state, color=colour)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel("Mean predicted probability of AI-generated")
ax.set_ylabel("Observed fraction AI-generated")
ax.set_title("External reliability before and after transferred scaling")
ax.grid(alpha=0.22)
ax.legend()
fig.tight_layout()
fig.savefig(RUN_DIR / "external_reliability_before_after.png", dpi=220)
plt.show()

effect_metrics = [
    "log_loss",
    "brier",
    "top_label_ece_15",
    "avg_conf_wrong",
    "hce_rate_all",
]
effect_frame = bootstrap_delta.set_index("metric").loc[effect_metrics].reset_index()
y = np.arange(len(effect_frame))
estimate = effect_frame["paired_delta_scaled_minus_raw"].to_numpy()
lower = estimate - effect_frame["delta_ci_2_5"].to_numpy()
upper = effect_frame["delta_ci_97_5"].to_numpy() - estimate
fig, ax = plt.subplots(figsize=(7.5, 4.8))
ax.axvline(0, color="#555555", linestyle="--", linewidth=1)
ax.errorbar(
    estimate,
    y,
    xerr=np.vstack([lower, upper]),
    fmt="o",
    color="#2A7F62",
    ecolor="#3F3F3F",
    capsize=4,
)
ax.set_yticks(y, effect_frame["metric"])
ax.set_xlabel("Paired delta: scaled minus raw (negative is improvement)")
ax.set_title("Temperature-scaling effects with paired bootstrap intervals")
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
fig.savefig(RUN_DIR / "paired_calibration_effect_forest.png", dpi=220)
plt.show()


## 9. Write a compact result summary and download the bundle

Keep the entire ZIP. It contains the prediction CSV, confidence intervals, policy tables,
figures, run configuration, and software versions needed to update the thesis.


In [ ]:
environment = {
    "python": sys.version,
    "platform": platform.platform(),
    "tensorflow": tf.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
    "scikit_learn": sklearn.__version__,
}
(RUN_DIR / "environment.json").write_text(json.dumps(environment, indent=2))

raw = overall_metrics[overall_metrics["state"] == "Raw"].iloc[0]
scaled = overall_metrics[overall_metrics["state"] == "Temperature-scaled"].iloc[0]
summary_lines = [
    "# External Open-Generator Reliability Improvement Run",
    "",
    f"- Dataset: `{DATASET_ID}` split `{SPLIT_TO_USE}`",
    f"- Sample: {len(predictions):,} images; `{subset_tag}`; seed {RANDOM_SEED}",
    f"- Temperature transferred from primary validation: {TEMPERATURE:.10f}",
    f"- Accuracy: {raw['accuracy']:.4f}",
    f"- Balanced accuracy: {raw['balanced_accuracy']:.4f}",
    f"- AUROC: {raw['auroc']:.4f}",
    f"- Specificity on external real images: {raw['specificity_real']:.4f}",
    f"- AI recall: {raw['ai_recall']:.4f}",
    f"- Top-label ECE-15: {raw['top_label_ece_15']:.4f} -> {scaled['top_label_ece_15']:.4f}",
    f"- High-confidence errors: {int(raw['high_confidence_errors'])} -> {int(scaled['high_confidence_errors'])}",
    "",
    "Interpretation boundary: this is an external cross-dataset open-generator stress test. "
    "It changes generator sources and real-image provenance and is not a causal estimate of generator novelty alone.",
]
summary_path = RUN_DIR / "run_summary.md"
summary_path.write_text("\n".join(summary_lines))
print(summary_path.read_text())

archive_path = shutil.make_archive(str(RUN_DIR), "zip", root_dir=RUN_DIR)
print("Saved ZIP:", archive_path)

if DOWNLOAD_ZIP_AT_END:
    from google.colab import files

    files.download(archive_path)
